# Metodología del Notebook 2 - Limpieza, transformación e integración de datos

## Objetivo

El objetivo del segundo notebook fue preparar una base analítica coherente a nivel de pedido, a partir de la limpieza y consolidación de las diferentes tablas del dataset. Esta fase buscó transformar los datos originales en una tabla maestra única, adecuada para el análisis exploratorio y para el posterior estudio del sentimiento en las reseñas.

## Procedimiento

En primer lugar, se generaron copias de trabajo de las tablas originales para preservar intactos los archivos de entrada. A continuación, se corrigieron nombres de columnas inconsistentes y se transformaron las variables temporales a formato `datetime`, especialmente en la tabla de pedidos y en las reseñas.

Posteriormente, se construyeron nuevas variables operativas relevantes para el negocio, como los días reales de entrega, el retraso respecto a la fecha estimada y una variable indicadora de entrega tardía. Estas variables permitieron trasladar la información logística a una forma directamente interpretable.

Dado que varias tablas no estaban a nivel de pedido, se realizó una agregación previa en `order_items`, `order_payments` y `order_reviews`. En el caso de `order_items`, se resumió cada pedido en métricas como número de artículos, número de vendedores, valor total y coste de envío. En `order_payments`, se calculó el número de pagos, el importe total pagado y el máximo número de cuotas. En `order_reviews`, se consolidó la información de puntuación y comentario a nivel de pedido.

También se integró la tabla de productos con la traducción de categorías y se preparó la geolocalización mediante una agregación por código postal, evitando unir la tabla geográfica en bruto. Finalmente, se realizó el merge de todas las fuentes sobre la tabla central de pedidos para construir una tabla maestra a nivel `order_id`.

## Justificación metodológica

La existencia de tablas con distinta granularidad hacía necesario un proceso de integración controlado. Si estas tablas se hubieran unido directamente sin agregación previa, se habrían duplicado pedidos y distorsionado indicadores básicos del negocio. Por ello, este notebook tuvo como finalidad garantizar la consistencia estructural de la base de análisis.

## Resultado esperado

El resultado de esta fase fue la creación de un `master_df` a nivel pedido, con variables de negocio, logística, cliente, categoría, geografía y satisfacción, listo para ser utilizado en el análisis exploratorio y en el análisis de sentimiento.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Carga de archivos:

In [ ]:
PATH = "../data/raw/"

customers = pd.read_csv(PATH + "olist_customers_dataset.csv")
geolocation = pd.read_csv(PATH + "olist_geolocation_dataset.csv")
order_items = pd.read_csv(PATH + "olist_order_items_dataset.csv")
order_payments = pd.read_csv(PATH + "olist_order_payments_dataset.csv")
order_reviews = pd.read_csv(PATH + "olist_order_reviews_dataset.csv")
orders = pd.read_csv(PATH + "olist_orders_dataset.csv")
products = pd.read_csv(PATH + "olist_products_dataset.csv")
sellers = pd.read_csv(PATH + "olist_sellers_dataset.csv")
category_translation = pd.read_csv(PATH + "product_category_name_translation.csv")

In [ ]:
customers_clean = customers.copy()
geolocation_clean = geolocation.copy()
order_items_clean = order_items.copy()
order_payments_clean = order_payments.copy()
order_reviews_clean = order_reviews.copy()
orders_clean = orders.copy()
products_clean = products.copy()
sellers_clean = sellers.copy()
category_translation_clean = category_translation.copy()

### Revisamos Shapes iniciales:

In [ ]:
clean_tables = {
    "customers_clean": customers_clean,
    "geolocation_clean": geolocation_clean,
    "order_items_clean": order_items_clean,
    "order_payments_clean": order_payments_clean,
    "order_reviews_clean": order_reviews_clean,
    "orders_clean": orders_clean,
    "products_clean": products_clean,
    "sellers_clean": sellers_clean,
    "category_translation_clean": category_translation_clean
}

for name, df in clean_tables.items():
    print(f"{name}: {df.shape}")

customers_clean: (99441, 5)
geolocation_clean: (1000163, 5)
order_items_clean: (112650, 7)
order_payments_clean: (103886, 5)
order_reviews_clean: (99224, 7)
orders_clean: (99441, 8)
products_clean: (32951, 9)
sellers_clean: (3095, 4)
category_translation_clean: (71, 2)


## Conversión de fechas en la tabla "orders"

In [ ]:
date_cols_orders = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for col in date_cols_orders:
    orders_clean[col] = pd.to_datetime(orders_clean[col], errors="coerce")

orders_clean[date_cols_orders].dtypes

order_purchase_timestamp         datetime64[ns]
order_approved_at                datetime64[ns]
order_delivered_carrier_date     datetime64[ns]
order_delivered_customer_date    datetime64[ns]
order_estimated_delivery_date    datetime64[ns]
dtype: object

## Conversión fechas en order_items y order_reviews

In [ ]:
order_items_clean["shipping_limit_date"] = pd.to_datetime(
    order_items_clean["shipping_limit_date"], errors="coerce"
)

review_date_cols = ["review_creation_date", "review_answer_timestamp"]

for col in review_date_cols:
    order_reviews_clean[col] = pd.to_datetime(order_reviews_clean[col], errors="coerce")

print(order_items_clean["shipping_limit_date"].dtype)
print(order_reviews_clean[review_date_cols].dtypes)

datetime64[ns]
review_creation_date       datetime64[ns]
review_answer_timestamp    datetime64[ns]
dtype: object


### Corrección de nombres en products

In [ ]:
products_clean = products_clean.rename(columns={
    "product_name_lenght": "product_name_length",
    "product_description_lenght": "product_description_length"
})

products_clean.columns.tolist()

['product_id',
 'product_category_name',
 'product_name_length',
 'product_description_length',
 'product_photos_qty',
 'product_weight_g',
 'product_length_cm',
 'product_height_cm',
 'product_width_cm']

## Miramos porcentaje de los nulos:

In [ ]:
critical_tables = {
    "orders_clean": orders_clean,
    "order_reviews_clean": order_reviews_clean,
    "products_clean": products_clean
}

for name, df in critical_tables.items():
    print(f"\n{name}")
    print((df.isnull().mean() * 100).sort_values(ascending=False))


orders_clean
order_delivered_customer_date   2.98
order_delivered_carrier_date    1.79
order_approved_at               0.16
order_id                        0.00
customer_id                     0.00
order_status                    0.00
order_purchase_timestamp        0.00
order_estimated_delivery_date   0.00
dtype: float64

order_reviews_clean
review_comment_title      88.34
review_comment_message    58.70
review_id                  0.00
order_id                   0.00
review_score               0.00
review_creation_date       0.00
review_answer_timestamp    0.00
dtype: float64

products_clean
product_category_name        1.85
product_name_length          1.85
product_description_length   1.85
product_photos_qty           1.85
product_weight_g             0.01
product_length_cm            0.01
product_height_cm            0.01
product_width_cm             0.01
product_id                   0.00
dtype: float64


### Creamos celdas para variables de tiempo de entrega:
El diccionario identifica estas columnas como parte del ciclo del pedido:
* 1 - Compra 
* 2 - Aprobación 
* 3 - Entrega al transportista
* 4 - Entrega al cliente 
* 5 - fecha estimada de entrega 

In [ ]:
orders_clean["delivery_days"] = (
    orders_clean["order_delivered_customer_date"] - orders_clean["order_purchase_timestamp"]
).dt.days

orders_clean["approval_days"] = (
    orders_clean["order_approved_at"] - orders_clean["order_purchase_timestamp"]
).dt.days

orders_clean["carrier_days"] = (
    orders_clean["order_delivered_carrier_date"] - orders_clean["order_purchase_timestamp"]
).dt.days

orders_clean["delay_days"] = (
    orders_clean["order_delivered_customer_date"] - orders_clean["order_estimated_delivery_date"]
).dt.days

orders_clean[[
    "order_purchase_timestamp",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
    "delivery_days",
    "delay_days"
]].head()

,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,delay_days
0,2017-10-02 10:56:33,2017-10-10 21:25:13,2017-10-18,8.00,-8.00
1,2018-07-24 20:41:37,2018-08-07 15:27:45,2018-08-13,13.00,-6.00
2,2018-08-08 08:38:49,2018-08-17 18:06:29,2018-09-04,9.00,-18.00
3,2017-11-18 19:28:06,2017-12-02 00:28:42,2017-12-15,13.00,-13.00
4,2018-02-13 21:18:39,2018-02-16 18:17:02,2018-02-26,2.00,-10.00


* delivery_days = order_delivered_customer_date - order_purchase_timestamp: Mide cuántos días pasaron desde que el cliente compró hasta que recibió el pedido.
* approval_days = order_approved_at - order_purchase_timestamp: Mide cuánto tardó la aprobación del pedido. Velocidad de validación/confirmación del pedido.
* carrier_days = order_delivered_carrier_date - order_purchase_timestamp: Mide cuánto pasó hasta que el pedido fue entregado al transportista. Rapidez operativa del inicio del envío.
* delay_days = order_delivered_customer_date - order_estimated_delivery_date: Compara la entrega real con la entrega prometida.
* .dt.days = Convierte la diferencia entre fechas en número de días.

In [ ]:
orders_clean["is_late"] = np.where(
    orders_clean["delay_days"].isna(),
    np.nan,
    np.where(orders_clean["delay_days"] > 0, 1, 0)
)

orders_clean["is_late"].value_counts(dropna=False)

is_late
0.00    89941
1.00     6535
NaN      2965
Name: count, dtype: int64

* Crea una variable binaria:
* 1 si delay_days > 0
0 si no
O sea:
1 = pedido tardío
0 = no tardío
NaN = no se puede saber 

value_counts(dropna=False)
Cuenta cuántos pedidos quedaron en cada grupo.

In [ ]:
order_items_agg = (
    order_items_clean
    .groupby("order_id", as_index=False)
    .agg(
        n_items=("order_item_id", "count"),
        n_products=("product_id", "nunique"),
        n_sellers=("seller_id", "nunique"),
        total_price=("price", "sum"),
        total_freight=("freight_value", "sum")
    )
)

order_items_agg.head()

,order_id,n_items,n_products,n_sellers,total_price,total_freight
0,00010242fe8c5a6d1ba2dd792cb16214,1,1,1,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,1,1,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,1,1,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,1,1,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,1,1,199.90,18.14


- n_items=("order_item_id", "count")
Cuenta cuántas líneas de artículo tiene el pedido.
Negocio: tamaño del pedido.
- n_products=("product_id", "nunique")
Cuenta cuántos productos distintos hay en el pedido.
Negocio: diversidad del pedido.
- n_sellers=("seller_id", "nunique")
Cuenta cuántos vendedores distintos participaron.
Negocio: complejidad logística del pedido.
- Esto es especialmente útil porque la guía dice que distintos artículos pueden ser enviados por vendedores distintos.
- total_price=("price", "sum")
Suma el precio de todos los artículos.
Negocio: valor de productos vendidos en ese pedido.
- total_freight=("freight_value", "sum")
Suma el coste de envío asociado a los artículos.
Negocio: coste logístico imputado al pedido.
head()

In [ ]:
order_payments_agg = (
    order_payments_clean
    .groupby("order_id", as_index=False)
    .agg(
        payment_count=("payment_sequential", "count"),
        payment_value_total=("payment_value", "sum"),
        max_installments=("payment_installments", "max")
    )
)

order_payments_agg.head()

,order_id,payment_count,payment_value_total,max_installments
0,00010242fe8c5a6d1ba2dd792cb16214,1,72.19,2
1,00018f77f2f0320c557190d7a144bdd3,1,259.83,3
2,000229ec398224ef6ca0657da4fc703e,1,216.87,5
3,00024acbcdf0a6daa1e931b038114c75,1,25.78,2
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,218.04,3


- La tabla order_payments puede tener varias filas por pedido, porque un pedido puede pagarse en varias secuencias o combinaciones. El diccionario confirma que esta tabla tiene payment_sequential, payment_installments y payment_value.
- groupby("order_id")
Agrupa todos los pagos de un mismo pedido.
.agg(...)
- Resume el comportamiento de pago en una sola fila por pedido.
Qué crea cada variable
- payment_count=("payment_sequential", "count")
Cuenta cuántas entradas de pago tiene el pedido.
Negocio: si el pedido tuvo una sola operación o varias.
- payment_value_total=("payment_value", "sum")
Suma el dinero total pagado.
Negocio: valor monetario final registrado en pagos.
- max_installments=("payment_installments", "max")
Toma el máximo número de cuotas del pedido.
Negocio: indicador simple de financiación o pago aplazado.

In [ ]:
order_reviews_clean.head()

,review_id,order_id,review_score,review_comment_title,review_comment_message,review_creation_date,review_answer_timestamp
0,7bc2406110b926393aa56f80a40eba40,73fc7af87114b39712e6da79b0a377eb,4,NaN,NaN,2018-01-18,2018-01-18 21:46:59
1,80e641a11e56f04c1ad469d5645fdfde,a548910a1c6147796b98fdf73dbeba33,5,NaN,NaN,2018-03-10,2018-03-11 03:05:13
2,228ce5500dc1d8e020d8d1322874b6f0,f9e4b658b201a9f2ecdecbb34bed034b,5,NaN,NaN,2018-02-17,2018-02-18 14:36:24
3,e64fb393e7b32834bb789ff8bb30750e,658677c97b385a9be170737859d3511b,5,NaN,Recebi bem antes do prazo estipulado.,2017-04-21,2017-04-21 22:02:06
4,f7c4243c7fe1938f181bec41a392bdeb,8e6bfb81e283fa7e4f11123a3fb894f1,5,NaN,Parabéns lojas lannister adorei comprar pela I...,2018-03-01,2018-03-02 10:26:53


In [ ]:
order_reviews_agg = (
    order_reviews_clean
    .groupby("order_id", as_index=False)
    .agg(
        review_score=("review_score", "mean"),
        review_creation_date=("review_creation_date", "max"),
        review_answer_timestamp=("review_answer_timestamp", "max"),
        review_comment_title=("review_comment_title", "first"),
        review_comment_message=("review_comment_message", "first")
    )
)

order_reviews_agg.head()

,order_id,review_score,review_creation_date,review_answer_timestamp,review_comment_title,review_comment_message
0,00010242fe8c5a6d1ba2dd792cb16214,5.00,2017-09-21,2017-09-22 10:57:03,None,"Perfeito, produto entregue antes do combinado."
1,00018f77f2f0320c557190d7a144bdd3,4.00,2017-05-13,2017-05-15 11:34:13,None,None
2,000229ec398224ef6ca0657da4fc703e,5.00,2018-01-23,2018-01-23 16:06:31,None,Chegou antes do prazo previsto e o produto sur...
3,00024acbcdf0a6daa1e931b038114c75,4.00,2018-08-15,2018-08-15 16:39:01,None,None
4,00042b26cf59d7ce69dfabb4e55b4fd9,5.00,2017-03-02,2017-03-03 10:54:59,None,Gostei pois veio no prazo determinado .


# Agrupamos por order_id y deja una sola fila por pedido.
Qué crea
- review_score: media de la puntuación si hubiera más de una fila
- fechas de review: toma la más reciente
- texto: toma el primer comentario disponible
- Impacto:
- Dejas la satisfacción del cliente lista para cruzarla luego con:
- 1 - retrasos
- 2 - categorías
- 3 - valor del pedido
- 4 - sentimiento.

## Preparar productos con traducción de categorías:
- Con el objetivo de preparar productos con traducción de categorías, une products_clean con la tabla de traducción.

- Aparecerá una nueva columna:
product_category_name_english

- Impacto:
Esto mejora muchísimo la lectura del análisis por categoría.



In [ ]:
products_with_category = products_clean.merge(
    category_translation_clean,
    on="product_category_name",
    how="left"
)

products_with_category.head()

,product_id,product_category_name,product_name_length,product_description_length,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,product_category_name_english
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1.00,225.00,16.00,10.00,14.00,perfumery
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.00,276.00,1.00,"1,000.00",30.00,18.00,20.00,art
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.00,250.00,1.00,154.00,18.00,9.00,15.00,sports_leisure
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.00,261.00,1.00,371.00,26.00,4.00,26.00,baby
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.00,402.00,4.00,625.00,20.00,17.00,13.00,housewares


 - Comparamos cuantas categorias quedaron sin traducir:


In [ ]:
products_with_category["product_category_name_english"].isnull().sum()

np.int64(623)

## Preparamos  geolocation

- geolocation tiene muchas filas repetidas por zip prefix.
No se debe unir en bruto.
- Primero hay que resumirla a nivel:
geolocation_zip_code_prefix

In [ ]:
geolocation_clean.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1037,-23.55,-46.64,sao paulo,SP
1,1046,-23.55,-46.64,sao paulo,SP
2,1046,-23.55,-46.64,sao paulo,SP
3,1041,-23.54,-46.64,sao paulo,SP
4,1035,-23.54,-46.64,sao paulo,SP


In [ ]:
geolocation_agg = (
    geolocation_clean
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "mean"),
        geolocation_lng=("geolocation_lng", "mean"),
        geolocation_city=("geolocation_city", "first"),
        geolocation_state=("geolocation_state", "first")
    )
)

geolocation_agg.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.55,-46.63,sao paulo,SP
1,1002,-23.55,-46.63,sao paulo,SP
2,1003,-23.55,-46.64,sao paulo,SP
3,1004,-23.55,-46.63,sao paulo,SP
4,1005,-23.55,-46.64,sao paulo,SP


- order_items está a nivel ítem, pero la tabla maestra final estará a nivel pedido.
Necesitamos una forma simple de representar la categoría del pedido.

In [ ]:
order_items_products = order_items_clean.merge(
    products_with_category[["product_id", "product_category_name", "product_category_name_english"]],
    on="product_id",
    how="left"
)

order_items_products.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,product_category_name_english
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,pet_shop
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87,moveis_decoracao,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79,perfumaria,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14,ferramentas_jardim,garden_tools


- Este ultimo añade la categoría del producto a cada línea de item.

In [ ]:
order_category = (
    order_items_products
    .groupby("order_id")["product_category_name_english"]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    .reset_index()
    .rename(columns={"product_category_name_english": "main_category"})
)

order_category.head()

,order_id,main_category
0,00010242fe8c5a6d1ba2dd792cb16214,cool_stuff
1,00018f77f2f0320c557190d7a144bdd3,pet_shop
2,000229ec398224ef6ca0657da4fc703e,furniture_decor
3,00024acbcdf0a6daa1e931b038114c75,perfumery
4,00042b26cf59d7ce69dfabb4e55b4fd9,garden_tools


- Busca la categoría más frecuente dentro de cada pedido, con la intención de dar una variable de categoría simple para análisis por pedido.

In [ ]:
customers_geo = customers_clean.merge(
    geolocation_agg,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

customers_geo.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,"14,409.00",-20.50,-47.40,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,"9,790.00",-23.73,-46.54,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,"1,151.00",-23.53,-46.66,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,"8,775.00",-23.50,-46.19,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,"13,056.00",-22.98,-47.14,campinas,SP


In [ ]:
customers_clean = customers.copy()
geolocation_clean = geolocation.copy()

## Agregación de geolocación

In [ ]:
geolocation_agg = (
    geolocation_clean
    .groupby("geolocation_zip_code_prefix", as_index=False)
    .agg(
        geolocation_lat=("geolocation_lat", "mean"),
        geolocation_lng=("geolocation_lng", "mean"),
        geolocation_city=("geolocation_city", "first"),
        geolocation_state=("geolocation_state", "first")
    )
)

geolocation_agg.head()

,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,1001,-23.55,-46.63,sao paulo,SP
1,1002,-23.55,-46.63,sao paulo,SP
2,1003,-23.55,-46.64,sao paulo,SP
3,1004,-23.55,-46.63,sao paulo,SP
4,1005,-23.55,-46.64,sao paulo,SP


## Agregación de geolocación

In [ ]:
customers_geo = customers_clean.merge(
    geolocation_agg,
    left_on="customer_zip_code_prefix",
    right_on="geolocation_zip_code_prefix",
    how="left"
)

customers_geo.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP,"14,409.00",-20.50,-47.40,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP,"9,790.00",-23.73,-46.54,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP,"1,151.00",-23.53,-46.66,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP,"8,775.00",-23.50,-46.19,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP,"13,056.00",-22.98,-47.14,campinas,SP


## Agregamos el merge final:

In [ ]:
master_df = (
    orders_clean
    .merge(customers_geo, on="customer_id", how="left")
    .merge(order_items_agg, on="order_id", how="left")
    .merge(order_payments_agg, on="order_id", how="left")
    .merge(order_reviews_agg, on="order_id", how="left")
    .merge(order_category, on="order_id", how="left")
)

master_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,delivery_days,approval_days,carrier_days,delay_days,is_late,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,geolocation_zip_code_prefix,geolocation_lat,geolocation_lng,geolocation_city,geolocation_state,n_items,n_products,n_sellers,total_price,total_freight,payment_count,payment_value_total,max_installments,review_score,review_creation_date,review_answer_timestamp,review_comment_title,review_comment_message,main_category
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,8.00,0.00,2.00,-8.00,0.00,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,"3,149.00",-23.58,-46.59,sao paulo,SP,1.00,1.00,1.00,29.99,8.72,3.00,38.71,1.00,4.00,2017-10-11,2017-10-12 03:43:48,None,"Não testei o produto ainda, mas ele veio corre...",housewares
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,13.00,1.00,1.00,-6.00,0.00,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,"47,813.00",-12.18,-44.66,barreiras,BA,1.00,1.00,1.00,118.70,22.76,1.00,141.46,1.00,4.00,2018-08-08,2018-08-08 18:37:50,Muito boa a loja,Muito bom o produto.,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,9.00,0.00,0.00,-18.00,0.00,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,"75,265.00",-16.75,-48.51,vianopolis,GO,1.00,1.00,1.00,159.90,19.22,1.00,179.12,3.00,5.00,2018-08-18,2018-08-22 19:07:58,None,None,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,13.00,0.00,3.00,-13.00,0.00,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,"59,296.00",-5.77,-35.27,sao goncalo do amarante,RN,1.00,1.00,1.00,45.00,27.20,1.00,72.20,1.00,5.00,2017-12-03,2017-12-05 19:21:58,None,O produto foi exatamente o que eu esperava e e...,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,2.00,0.00,0.00,-10.00,0.00,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,"9,195.00",-23.68,-46.51,santo andre,SP,1.00,1.00,1.00,19.90,8.72,1.00,28.62,1.00,5.00,2018-02-17,2018-02-18 13:02:51,None,None,stationery


In [ ]:
type(customers_geo)

pandas.core.frame.DataFrame

## Hacemos revisión de columnas claves:

In [ ]:
master_df[[
    "order_id",
    "customer_unique_id",
    "order_status",
    "delivery_days",
    "delay_days",
    "is_late",
    "n_items",
    "total_price",
    "payment_value_total",
    "review_score",
    "main_category"
]].head()

,order_id,customer_unique_id,order_status,delivery_days,delay_days,is_late,n_items,total_price,payment_value_total,review_score,main_category
0,e481f51cbdc54678b7cc49136f2d6af7,7c396fd4830fd04220f754e42b4e5bff,delivered,8.00,-8.00,0.00,1.00,29.99,38.71,4.00,housewares
1,53cdb2fc8bc7dce0b6741e2150273451,af07308b275d755c9edb36a90c618231,delivered,13.00,-6.00,0.00,1.00,118.70,141.46,4.00,perfumery
2,47770eb9100c2d0c44946d9cf07ec65d,3a653a41f6f9fc3d2a113cf8398680e8,delivered,9.00,-18.00,0.00,1.00,159.90,179.12,5.00,auto
3,949d5b44dbf5de918fe9c16f97b45f8a,7c142cf63193a1473d2e66489a9ae977,delivered,13.00,-13.00,0.00,1.00,45.00,72.20,5.00,pet_shop
4,ad21c59c0840e6cb83a9ceb5573f8159,72632f0f9dd73dfee390c9b22eb56dd6,delivered,2.00,-10.00,0.00,1.00,19.90,28.62,5.00,stationery


In [ ]:
master_df.to_csv("../data/final/master_orders_dataset.csv", index=False)

### Hacemos una pequeña comprobación de los valores:

In [ ]:
master_df.shape

(99441, 36)

In [ ]:
master_df["order_id"].nunique(), len(master_df)

(99441, 99441)

In [ ]:
master_df[[
    "customer_unique_id",
    "n_items",
    "payment_value_total",
    "review_score",
    "main_category"
]].isnull().mean().sort_values(ascending=False) * 100

main_category         2.20
n_items               0.78
review_score          0.77
payment_value_total   0.00
customer_unique_id    0.00
dtype: float64

In [ ]:
master_df[[
    "order_id",
    "customer_unique_id",
    "order_status",
    "delivery_days",
    "delay_days",
    "is_late",
    "n_items",
    "total_price",
    "payment_value_total",
    "review_score",
    "main_category",
    "geolocation_state"
]].sample(10, random_state=42)

,order_id,customer_unique_id,order_status,delivery_days,delay_days,is_late,n_items,total_price,payment_value_total,review_score,main_category,geolocation_state
52263,b9a6c5f5df52c7226ac85aee7524c27f,2d1bf256227e4d22d10ea6c0b81809d7,delivered,6.00,-28.00,0.00,1.00,99.00,117.79,5.00,health_beauty,RS
46645,261e71d2349c713eafa9f3df5972b95d,12bf514b8d413d8cbe66a2665f4b724c,delivered,9.00,-16.00,0.00,1.00,157.77,172.62,5.00,NaN,MG
37546,67b50899f52995848c427e361e10dde3,83c6df0d47130de38c99cebe96521e8a,delivered,10.00,-19.00,0.00,1.00,90.00,108.43,1.00,auto,PR
94756,32733fc014b67ef70fa6039dd8c6ba82,29b186723b197669f69b7d63c3e27c07,delivered,25.00,3.00,1.00,1.00,229.99,256.05,5.00,cool_stuff,RJ
14771,39a70e9e9b729b11dee34ac12478597f,a59129ed35da4c3e3f2a005b4c6582fc,delivered,11.00,-21.00,0.00,1.00,108.90,128.90,3.00,sports_leisure,PR
36263,80000ae9d118d79953522e35cce34f13,c9fff85f2617b8e1a78142aa11ee962f,delivered,8.00,-12.00,0.00,2.00,99.80,121.72,4.00,furniture_decor,SP
98556,2bfd14409ba8ba1153ce42b2abc44bb8,774efd8e5cc822162e0af7311d81f5c1,delivered,17.00,-13.00,0.00,1.00,152.00,167.81,5.00,fashion_underwear_beach,GO
23747,faa01b7a24d0ba51d0d0ec652cd9b745,7a76f872eec388119a13858df1571b2b,delivered,14.00,-12.00,0.00,1.00,41.89,69.13,4.00,costruction_tools_garden,BA
50315,06aa0aa8c22027ebb953d501c4238512,edcc7ce9b67e904a02c89576d13e1ada,delivered,1.00,-14.00,0.00,1.00,69.90,78.78,4.00,bed_bath_table,SP
6501,3762c0ef92b5bc397671e6d7e6d5672f,cdc6cd6f0a8b8f844db9a61e779055b3,delivered,7.00,-21.00,0.00,1.00,59.90,73.34,5.00,garden_tools,SP


In [ ]:
(master_df.isnull().mean() * 100).sort_values(ascending=False).head(15)

review_comment_title            88.38
review_comment_message          58.93
order_delivered_customer_date    2.98
delivery_days                    2.98
delay_days                       2.98
is_late                          2.98
main_category                    2.20
order_delivered_carrier_date     1.79
carrier_days                     1.79
total_price                      0.78
n_items                          0.78
n_products                       0.78
n_sellers                        0.78
total_freight                    0.78
review_score                     0.77
dtype: float64

- review_comment_title            88.38
- review_comment_message          58.93
##### Esto significa que mucha gente deje solo la puntuación (review_score) y no escriba título ni comentario


- order_delivered_customer_date, delivery_days, delay_days, is_late = 2.98%
Esto también es normal.
Qué significa
-  Hay pedidos donde no existe entrega final registrada, normalmente por casos como:
- 1 - cancelados
- 2 - no entregados
- 3 - otras situaciones del flujo del pedido



In [ ]:
delivery_df = master_df[master_df["order_status"] == "delivered"].copy()

In [ ]:
master_df["order_id"].nunique(), len(master_df)

(99441, 99441)

### Validación final:

- (99441, 99441) significa:

- 1 - order_id únicos = 99,441

- 2 - total de filas = 99,441